# Train an ML Model

In this notebook we train a linear regression model on the NYC Green Taxi dataset and track the run with MLflow. The goal is to create a small historical baseline that we can later monitor for drift.

We will use one month of data, a small feature set, and a local MLflow tracking store so the project can stay runnable on a single machine.

This is the starting notebook for the MLOps project, adapted from [mle-monitoring-in-ml](https://github.com/neuefische/mle-monitoring-in-ml/blob/2a805a98f7d6cf2266b291c0984c98f93bc6d9d3/01-train-ml-model.ipynb). Run it once to establish the baseline, then follow [project-description.md](project-description.md) to move preprocessing, training, and evaluation into reusable Python code.

This is a historical replay: the [TLC data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_green.pdf) defines `trip_distance` as elapsed taximeter distance. It is not an estimated distance available at pickup.


In [ ]:
import logging
import os
from pathlib import Path
from urllib.request import urlretrieve

import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

# Reduce utility log noise without hiding warnings about model packaging.
logging.getLogger("mlflow.utils").setLevel(logging.WARNING)

In [ ]:
year = 2024
month = 1
color = "green"

In [ ]:
# Download the data
data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

In [ ]:
# Load the data
df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
df.shape

Now we will connect the notebook to MLflow. For a local-first setup, copy `.env.example` to `.env` if you haven't done it yet and keep `MLFLOW_TRACKING_URI=sqlite:///mlflow.db` unless you already have an MLflow server running elsewhere.

You can also start the MLflow UI from the repository root with:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```


We will create or reuse an experiment called `green-taxi-monitoring` in MLflow. We also add a small tag set so the baseline run is easier to identify when comparing it with your refactored training code.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

experiment_name = "green-taxi-monitoring"
exp = mlflow.get_experiment_by_name(experiment_name)

if exp is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        tags={"developer": "<your-name>"},
    )
    print(f"Created experiment: {experiment_name} (ID: {experiment_id})")
else:
    print(f"Using existing experiment: {experiment_name} (ID: {exp.experiment_id})")

mlflow.set_experiment(experiment_name)

If everything went well, you should now be able to inspect the experiment in the MLflow UI. With the local setup above, that UI is available at <http://127.0.0.1:5000>.


Let's start now by looking at the data:

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Look for missing values
df.isnull().sum()

None of the columns we need have missing values. To predict the duration of a trip, we will use:

- `PULocationID`: the pickup location ID
- `DOLocationID`: the dropoff location ID
- `trip_distance`: the distance of the trip in miles

First we calculate trip duration in minutes from the Green Taxi pickup and dropoff timestamps: `lpep_pickup_datetime` and `lpep_dropoff_datetime`. We also remove trips shorter than 1 minute or longer than 60 minutes, and trips shorter than 0.1 miles or longer than 100 miles, so the baseline is not dominated by obvious outliers. Finally, we store the location IDs as strings: they are zone labels, not quantities, so the model should treat each zone as its own category.

In [ ]:
features = ["PULocationID", "DOLocationID", "trip_distance"]
target = "duration"

In [ ]:
# Calculate duration, remove outliers, and treat the location IDs as categories
def preprocess(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[df["trip_distance"].between(0.1, 100)]
    df[["PULocationID", "DOLocationID"]] = df[["PULocationID", "DOLocationID"]].astype(
        str
    )
    df = df[features + [target]]
    return df

In [ ]:
df_processed = preprocess(df)

In [ ]:
df_processed.head()

In [ ]:
df_processed.duration.describe()

In [ ]:
df_processed.duration.hist()

Now that we have the DataFrame that we want to train our model on, we need to split it into a train and test set. We will use 80% of the data for training and 20% for testing.

In [ ]:
y = df_processed["duration"]
X = df_processed.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

And now we can train the model and track the run with MLflow. The model is a scikit-learn pipeline with two steps:

1. `DictVectorizer` one-hot encodes the pickup and dropoff zones (one column per zone) and passes `trip_distance` through unchanged.
2. `LinearRegression` learns how much each zone and each mile adds to the trip duration.

`DictVectorizer` expects one dictionary per trip, so we convert the DataFrames with `to_dict(orient="records")` before training and predicting.

We will set a few tags to make the run easier to find later.

- `model`: `linear regression`
- `developer`: `<your-name>`
- `dataset`: `green-taxi`
- `year` and `month`: which batch produced the training data
- `features`: the columns used for training
- `target`: the value we want to predict

We will also log the evaluation metric and register the trained model under a stable name so your project API can load it again. We register the model from the URI returned by `mlflow.sklearn.log_model(...)`.

In [ ]:
with mlflow.start_run():
    tags = {
        "model": "linear regression",
        "developer": "<your-name>",
        "dataset": f"{color}-taxi",
        "year": year,
        "month": month,
        "features": features,
        "target": target,
    }
    mlflow.set_tags(tags)

    pipeline = make_pipeline(DictVectorizer(), LinearRegression())
    pipeline.fit(X_train.to_dict(orient="records"), y_train)

    y_pred = pipeline.predict(X_test.to_dict(orient="records"))
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline, name="mlflow-model", serialization_format="skops"
    )
    active_run = mlflow.active_run()
    assert active_run is not None
    run_id = active_run.info.run_id

    client = MlflowClient()
    model_name = "green-taxi-ride-duration"

    try:
        client.create_registered_model(
            name=model_name,
            tags={"framework": "scikit-learn"},
            description="Model to predict ride duration in NYC",
        )
    except MlflowException as exc:
        # Reuse the model only if it already exists; re-raise any other MLflow error.
        if exc.error_code != "RESOURCE_ALREADY_EXISTS":
            raise
        print(f"Registered model {model_name} already exists. Reusing it.")

    mv = client.create_model_version(
        name=model_name,
        source=model_info.model_uri,
        run_id=run_id,
    )

    client.set_registered_model_alias(
        name=model_name,
        version=mv.version,
        alias="production",
    )

You should now see your run in the MLflow UI. Click the run to see the logged tags, the metric, and the saved model.

![MLflow runs list](assets/01-monitoring_mlflow_registered_model.png)

The next cell confirms which model version now holds the `production` alias. When you refactor this notebook, keep the feature handling, evaluation split, and model-loading contract reproducible.


In [ ]:
print(f"Registered model: {model_name}, version {mv.version}, alias: production")